In [ ]:
# ============================================
# BATCH ORTHODONTIC SUITABILITY CHECK FOR ALL PATIENTS
# ============================================

import torch
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom
import os

def check_orthodontic_suitability(cbct_path, model_path='/content/drive/MyDrive/MMDental/models/best_model.pth'):
    """
    Simple check if patient is suitable for orthodontic treatment
    Rule: ANY disease with probability > 25% = NOT suitable
    """

    # Load model
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

    class Simple3DCNN(torch.nn.Module):
        def __init__(self, num_classes=8):
            super().__init__()
            self.conv_layers = torch.nn.Sequential(
                torch.nn.Conv3d(1, 32, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(32),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(32, 64, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(64),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(64, 128, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(128),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(128, 256, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(256),
                torch.nn.ReLU(inplace=True),
                torch.nn.AdaptiveAvgPool3d((1, 1, 1))
            )
            self.classifier = torch.nn.Sequential(
                torch.nn.Dropout(0.5),
                torch.nn.Linear(256, 128),
                torch.nn.ReLU(inplace=True),
                torch.nn.Dropout(0.3),
                torch.nn.Linear(128, num_classes)
            )

        def forward(self, x):
            x = self.conv_layers(x)
            x = x.view(x.size(0), -1)
            return self.classifier(x)

    model = Simple3DCNN(num_classes=len(checkpoint['class_names']))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Load and preprocess CBCT
    try:
        nifti = nib.load(cbct_path)
        volume = nifti.get_fdata().astype(np.float32)
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

        factors = (64/volume.shape[0], 64/volume.shape[1], 64/volume.shape[2])
        volume = zoom(volume, factors, order=1)
        volume = np.expand_dims(volume, axis=(0, 1))
        volume = torch.from_numpy(volume).float()

        # Predict
        with torch.no_grad():
            outputs = model(volume)
            probs = torch.nn.functional.softmax(outputs, dim=1)

        probs = probs.numpy()[0]
        class_names = checkpoint['class_names']

        # Check for any disease above 20%
        diseases_above_threshold = []
        for i, (condition, prob) in enumerate(zip(class_names, probs)):
            if prob > 0.25:
                diseases_above_threshold.append((condition, prob))

        return {
            'success': True,
            'suitable': len(diseases_above_threshold) == 0,
            'diseases': diseases_above_threshold,
            'all_probabilities': dict(zip(class_names, probs))
        }
    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }


# ============================================
# RUN FOR ALL PATIENTS 1 TO 658
# ============================================

base_path = '/content/drive/MyDrive/MMDental/MMDental'
model_path = '/content/drive/MyDrive/MMDental/models/best_model.pth'

print("\n" + "="*80)
print("BATCH ORTHODONTIC SUITABILITY ASSESSMENT")
print("Patients 1 to 658")
print("="*80)

results = []
suitable_count = 0
unsuitable_count = 0
not_found_count = 0

for patient_id in range(1, 659):
    cbct_path = f'{base_path}/{patient_id}/{patient_id}.nii.gz'

    # Check if file exists
    if not os.path.exists(cbct_path):
        print(f"❌ Patient {patient_id}: CBCT file not found - skipping")
        not_found_count += 1
        results.append({
            'patient_id': patient_id,
            'status': 'NOT_FOUND',
            'suitable': False
        })
        continue

    # Run prediction
    result = check_orthodontic_suitability(cbct_path, model_path)

    if result['success']:
        if result['suitable']:
            suitable_count += 1
            status = "✅ SUITABLE"
            diseases_text = "None"
        else:
            unsuitable_count += 1
            status = "❌ NOT SUITABLE"
            diseases_text = ', '.join([f"{d[0]}({d[1]:.0%})" for d in result['diseases']])

        print(f"Patient {patient_id:3d}: {status} - {diseases_text}")

        results.append({
            'patient_id': patient_id,
            'status': 'SUCCESS',
            'suitable': result['suitable'],
            'diseases': result.get('diseases', []),
            'probabilities': result.get('all_probabilities', {})
        })
    else:
        print(f"⚠️  Patient {patient_id}: Error - {result.get('error', 'Unknown error')}")
        not_found_count += 1
        results.append({
            'patient_id': patient_id,
            'status': 'ERROR',
            'suitable': False,
            'error': result.get('error', 'Unknown')
        })

# ============================================
# SUMMARY REPORT
# ============================================

print("\n" + "="*80)
print("SUMMARY REPORT")
print("="*80)

print(f"\n📊 Total patients processed: {len(results)}")
print(f"   ✅ Suitable for orthodontics: {suitable_count}")
print(f"   ❌ Not suitable (need treatment): {unsuitable_count}")
print(f"   ⚠️  Not found/Error: {not_found_count}")

if suitable_count > 0:
    print(f"\n📈 SUITABILITY RATE: {suitable_count/(suitable_count+unsuitable_count)*100:.1f}% of patients with CBCT")

# List suitable patients (first 20 only to avoid clutter)
suitable_patients = [r for r in results if r.get('suitable', False)]
if suitable_patients:
    print(f"\n✅ PATIENTS SUITABLE FOR ORTHODONTICS (first 20 shown):")
    for r in suitable_patients[:20]:
        print(f"   • Patient {r['patient_id']}")
    if len(suitable_patients) > 20:
        print(f"   ... and {len(suitable_patients) - 20} more")

# List unsuitable patients with their diseases (first 20)
unsuitable_patients = [r for r in results if not r.get('suitable', False) and r.get('status') == 'SUCCESS']
if unsuitable_patients:
    print(f"\n❌ PATIENTS REQUIRING TREATMENT FIRST (first 20 shown):")
    for r in unsuitable_patients[:20]:
        diseases = ', '.join([d[0] for d in r.get('diseases', [])])
        print(f"   • Patient {r['patient_id']}: {diseases}")
    if len(unsuitable_patients) > 20:
        print(f"   ... and {len(unsuitable_patients) - 20} more")

# ============================================
# EXPORT RESULTS TO CSV
# ============================================

import pandas as pd

# Create DataFrame for export
export_data = []
for r in results:
    if r['status'] == 'SUCCESS':
        row = {
            'patient_id': r['patient_id'],
            'suitable_for_orthodontics': r['suitable'],
            'diseases_detected': ', '.join([d[0] for d in r.get('diseases', [])]),
            'primary_disease': r.get('diseases', [])[0][0] if r.get('diseases') else 'None'
        }
        # Add individual disease probabilities
        if 'probabilities' in r:
            for disease, prob in r['probabilities'].items():
                row[f'prob_{disease.replace(" ", "_")}'] = f'{prob:.1%}'
    else:
        row = {
            'patient_id': r['patient_id'],
            'suitable_for_orthodontics': False,
            'diseases_detected': 'FILE_NOT_FOUND',
            'primary_disease': 'ERROR'
        }
    export_data.append(row)

df_results = pd.DataFrame(export_data)
csv_path = '/content/drive/MyDrive/MMDental/orthodontic_suitability_report.csv'
df_results.to_csv(csv_path, index=False)
print(f"\n📁 Full results exported to: {csv_path}")

# ============================================
# QUICK STATISTICS
# ============================================

print("\n" + "="*80)
print("QUICK STATISTICS")
print("="*80)

if suitable_count + unsuitable_count > 0:
    print(f"""
    ╔══════════════════════════════════════════════════════════════╗
    ║  CLINICAL SUMMARY                                            ║
    ╠══════════════════════════════════════════════════════════════╣
    ║  Total patients with CBCT: {suitable_count + unsuitable_count}                                    ║
    ║  Ready for orthodontics: {suitable_count} ({suitable_count/(suitable_count+unsuitable_count)*100:.0f}%)                             ║
    ║  Need dental treatment first: {unsuitable_count} ({unsuitable_count/(suitable_count+unsuitable_count)*100:.0f}%)                       ║
    ╚══════════════════════════════════════════════════════════════╝
    """)

# ============================================
# OPTIONAL: SAVE SUITABLE PATIENT LIST
# ============================================

suitable_ids = [r['patient_id'] for r in results if r.get('suitable', False)]
if suitable_ids:
    with open('/content/drive/MyDrive/MMDental/suitable_for_orthodontics.txt', 'w') as f:
        f.write('\n'.join(map(str, suitable_ids)))
    print(f"📋 List of suitable patients saved to: suitable_for_orthodontics.txt")
    print(f"   First 10 suitable patients: {', '.join(map(str, suitable_ids[:10]))}")

print("\n" + "="*80)
print("✅ BATCH ASSESSMENT COMPLETE!")
print("="*80)